# 03 — Sensitivity paths: when the budget is the object

A single bound at \(\Lambda=3\) is useful. Stakeholders often ask a harder
question: **how does the answer change as we relax the confounding limit?**

This notebook stays with the job-training extract and treats the sensitivity
parameter as the analysis object: paths, breakdown points, and short grids.


## Setup


In [1]:
from pathlib import Path

def tutorial_data(name: str) -> Path:
    """Resolve notebooks/data/<name> whether __file__ is set (CI) or not."""
    here = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    candidates = [
        here / "data" / name,
        here / "notebooks" / "data" / name,
        Path.cwd() / "notebooks" / "data" / name,
        Path.cwd() / "data" / name,
    ]
    for p in candidates:
        if p.is_file():
            return p
    raise FileNotFoundError(f"could not find notebooks/data/{name}; tried {candidates}")

import pandas as pd
from partialid import Assumptions, Query, bound, bound_grid, bound_path, from_dataframe

df = pd.read_csv(tutorial_data("job_training.csv"))
data = from_dataframe(df, columns=["treated", "employed"])
Q = Query.interventional(outcome="employed", do={"treated": 1}, value=1)
base = Assumptions.manski("treated", "employed")
print(Q.describe())


P(employed=1 | do(treated=1))


## Fixed budget reminder


In [2]:
r = bound(data.variables, data.P, Q, Assumptions.sensitivity_msm("treated", "employed", 3.0))
print(r.summary(Q))
print(
    "At Λ=3, universal-treatment employment lies in "
    f"[{r.lower:.3f}, {r.upper:.3f}] under Manski + Tan MSM."
)


query: P(employed=1 | do(treated=1))
interval: [0.473234, 0.809234]
width: 0.336
method: msm_agg/msm_margin_binary_xy
tight: True
success: True
message: closed margin
At Λ=3, universal-treatment employment lies in [0.473, 0.809] under Manski + Tan MSM.


## Path in \(\Lambda\)

`bound_path` sweeps a budget parameter. For non-default treatment names, pass
`treatment=` explicitly (it defaults to `"X"`).


In [3]:
path = bound_path(
    data.variables, data.P, Q, base,
    param="Lambda",
    treatment="treated",
    theta_min=1.0,
    theta_max=5.0,
    n_grid=9,
)
print(path.summary(at=3.0))
print()
print(f"{'Λ':>6}  {'lower':>8}  {'upper':>8}  {'width':>8}")
for lam in (1.0, 1.5, 2.0, 3.0, 4.0, 5.0):
    lo, hi, w = path.at(lam)
    print(f"{lam:6g}  {lo:8.4f}  {hi:8.4f}  {w:8.4f}")


sensitivity path in Lambda
range: [1, 5]
closed_form: False
width @ endpoints: 0 → 0.4032
at Lambda=3: [0.473234, 0.809234] (width 0.336)

     Λ     lower     upper     width
     1    0.7127    0.7127    0.0000
   1.5    0.6403    0.7610    0.1207
     2    0.5679    0.7851    0.2172
     3    0.4732    0.8092    0.3360
     4    0.4433    0.8213    0.3780
     5    0.4253    0.8285    0.4032


At \(\Lambda=1\) the MSM forces no unobserved confounding on this margin
(width \(\to 0\) here). As \(\Lambda\) grows, the interval opens toward the
Manski width — the sensitivity curve makes that tradeoff visible.


## Breakdown: when does a threshold enter the interval?

`path.breakdown(null)` returns the smallest budget at which `null` lies inside
the bound interval (or `None` if it never does in the searched range).

Here we ask when the interval for universal-treatment employment expands enough
to include 0.50 — a round policy threshold an analyst might care about.


In [4]:
assoc = float(df.loc[df["treated"] == 1, "employed"].mean())
print(f"associational employment among treated = {assoc:.4f}")

for null in (0.50, 0.55, assoc):
    b = path.breakdown(null)
    print(f"null={null:.4f} enters at Λ ≈ {b}")


associational employment among treated = 0.7127
null=0.5000 enters at Λ ≈ 2.468950084099136
null=0.5500 enters at Λ ≈ 2.1236423703703697
null=0.7127 enters at Λ ≈ 1.0


## Stricter path under MTR

If you also believe monotone treatment response, the whole sensitivity curve
shifts. Compose MTR into the **base** assumptions passed to `bound_path`.


In [5]:
base_mtr = base & Assumptions.mtr("employed", "treated")
path_mtr = bound_path(
    data.variables, data.P, Q, base_mtr,
    param="Lambda",
    treatment="treated",
    theta_min=1.0,
    theta_max=5.0,
    n_grid=9,
)
print(f"{'Λ':>6}  {'Manski+MSM width':>16}  {'+MTR width':>12}")
for lam in (1.0, 2.0, 3.0, 5.0):
    _, _, w0 = path.at(lam)
    _, _, w1 = path_mtr.at(lam)
    print(f"{lam:6g}  {w0:16.4f}  {w1:12.4f}")


     Λ  Manski+MSM width    +MTR width
     1            0.0000        0.0000
     2            0.2172        0.2172
     3            0.3360        0.2517
     5            0.4032        0.2710


## Other budget parameters

The same machinery works for Rosenbaum \(\Gamma\) paths and for discrete grids
in \(c\) or \(\kappa\) via `bound_grid`.


In [6]:
path_g = bound_path(
    data.variables, data.P, Q, base,
    param="Gamma",
    treatment="treated",
    theta_min=1.0,
    theta_max=6.0,
    n_grid=7,
)
print("Gamma path:")
for g in (1.0, 2.0, 4.0, 6.0):
    lo, hi, w = path_g.at(g)
    print(f"  Γ={g:g}: [{lo:.4f}, {hi:.4f}]  width={w:.4f}")

print("\nc-dependence grid:")
grid = bound_grid(
    data.variables, data.P, Q, base,
    param="c",
    values=[0.0, 0.05, 0.1, 0.2],
    treatment="treated",
)
for c, rr in grid:
    print(f"  c={c:g}: [{rr.lower:.4f}, {rr.upper:.4f}]  width={rr.width:.4f}")


Gamma path:
  Γ=1: [0.7127, 0.7127]  width=0.0000
  Γ=2: [0.6527, 0.7551]  width=0.1024
  Γ=4: [0.5679, 0.7851]  width=0.2172
  Γ=6: [0.5028, 0.7984]  width=0.2956

c-dependence grid:
  c=0: [0.7127, 0.7127]  width=0.0000
  c=0.05: [0.6805, 0.7390]  width=0.0585
  c=0.1: [0.6402, 0.7609]  width=0.1208
  c=0.2: [0.5186, 0.7953]  width=0.2767


## How to report a sensitivity analysis

Example sentence you can adapt:

> Under order-only Manski assumptions and Tan’s marginal sensitivity model,
> the employment rate under universal treatment lies in
> \([\ell(\Lambda), u(\Lambda)]\). At \(\Lambda=3\) the interval is
> \([\ldots]\). The conclusion that the rate exceeds 0.50 is robust for
> \(\Lambda\) up to approximately \(\Lambda^\star\) (breakdown). Adding MTR
> further narrows the path.

Always state the query, the assumption list (`a.describe()`), and whether
`tight` is true.

**Next:** [04_instruments_and_graphs.ipynb](04_instruments_and_graphs.ipynb) —
encouragement designs, IV recipes, and explicit diagrams.
